# Gold Layer: Product & Category Performance Analytics

This notebook calculates:
1. Product-level sales volume, gross revenue, average prices, and intra-category rankings.
2. Category-level aggregate volume, revenue, and Pareto cumulative revenue contribution (80/20 analysis).

Materialized as Delta Lake tables under `workspace.ecompulse_gold`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
products = spark.read.table("workspace.ecompulse_silver.products")
order_items = spark.read.table("workspace.ecompulse_silver.order_items")
orders = spark.read.table("workspace.ecompulse_silver.orders")

In [0]:
# Join order_items with products and orders at item grain
joined_df = (
    order_items
    .join(products.select("product_id", "product_category_name"), on="product_id", how="inner")
    .join(
        orders.select("order_id", "order_purchase_date", "order_purchase_timestamp", "order_status"),
        on="order_id",
        how="inner"
    )
    .fillna({"product_category_name": "Unknown"})
)

In [0]:
# Product-level aggregation
product_agg = (
    joined_df
    .groupBy("product_id", "product_category_name")
    .agg(
        F.count("order_item_id").alias("units_sold"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.countDistinct("order_id").alias("distinct_orders_count"),
        F.round(F.sum("freight_value"), 2).alias("total_freight"),
        F.min("order_purchase_date").alias("first_order_date"),
        F.max("order_purchase_date").alias("last_order_date")
    )
)

In [0]:
# Window functions for product performance
category_rank_window = Window.partitionBy("product_category_name").orderBy(F.col("total_revenue").desc())
overall_rank_window = Window.orderBy(F.col("total_revenue").desc())

product_performance = (
    product_agg
    .withColumn("category_dense_rank", F.dense_rank().over(category_rank_window))
    .withColumn("overall_rank", F.rank().over(overall_rank_window))
    .withColumn("orders_count", F.col("distinct_orders_count"))
    .withColumn("gross_sales_revenue", F.col("total_revenue"))
    .withColumn("avg_unit_price", F.col("avg_price"))
    .withColumn("category_rank_by_revenue", F.col("category_dense_rank"))
)

In [0]:
# Persist product_performance to Delta table
product_performance.write.mode("overwrite").format("delta").saveAsTable("workspace.ecompulse_gold.product_performance")

In [0]:
%sql
SELECT * FROM workspace.ecompulse_gold.product_performance ORDER BY total_revenue DESC LIMIT 5;

In [0]:
# Category-level aggregation
category_agg = (
    joined_df
    .groupBy("product_category_name")
    .agg(
        F.countDistinct("product_id").alias("distinct_products_count"),
        F.count("order_item_id").alias("total_units_sold"),
        F.round(F.sum("price"), 2).alias("total_category_revenue"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.countDistinct("order_id").alias("distinct_orders_count"),
        F.round(F.sum("freight_value"), 2).alias("total_freight"),
        F.min("order_purchase_date").alias("first_order_date"),
        F.max("order_purchase_date").alias("last_order_date")
    )
)

In [0]:
# Window functions for category performance & Pareto cumulative percentage
pareto_running_window = (
    Window.orderBy(F.col("total_category_revenue").desc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
pareto_total_window = Window.partitionBy()
category_rank_window = Window.orderBy(F.col("total_category_revenue").desc())

category_performance = (
    category_agg
    .withColumn("running_category_revenue", F.sum("total_category_revenue").over(pareto_running_window))
    .withColumn("all_categories_revenue", F.sum("total_category_revenue").over(pareto_total_window))
    .withColumn(
        "pareto_revenue_pct",
        F.when(F.col("all_categories_revenue") == 0, 0.0)
         .otherwise(F.round((F.col("running_category_revenue") / F.col("all_categories_revenue")) * 100.0, 2))
    )
    .withColumn("category_revenue_rank", F.dense_rank().over(category_rank_window))
    .withColumn("units_sold", F.col("total_units_sold"))
    .withColumn("category_revenue", F.col("total_category_revenue"))
    .withColumn("cumulative_revenue_pct", F.col("pareto_revenue_pct"))
    .withColumn("orders_count", F.col("distinct_orders_count"))
    .withColumn("avg_item_price", F.col("avg_price"))
    .drop("running_category_revenue", "all_categories_revenue")
)

In [0]:
# Persist category_performance to Delta table
category_performance.write.mode("overwrite").format("delta").saveAsTable("workspace.ecompulse_gold.category_performance")

In [0]:
%sql
SELECT * FROM workspace.ecompulse_gold.category_performance ORDER BY total_category_revenue DESC LIMIT 5;